<a href="https://colab.research.google.com/github/boruizhang/representations/blob/main/06_LLM_assisted_Post_OCR_Correction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM-Assisted Post-OCR Correction

## What is OCR?

OCR (Optical Character Recognition) is the process of converting images of text — such as scanned documents, photographs of printed pages, or PDFs — into machine-readable text. While OCR works well on clean, modern documents, it often introduces errors when processing older or degraded materials: characters get misread (e.g., "rn" becomes "m"), words get garbled, and formatting breaks down. Post-OCR correction is the step where we clean up these errors, and LLMs can help automate much of that work.

## About This Exercise

We'll be processing OCR'd scans of historical legal documents (1959 Trinidad and Tobago Constitution Amendment Order in Council), available at the UF dLoc Collection on Caribbean and North Atlantic territories:
https://dloc.com/UF00085556/00001/citation

A basic **Post-OCR Error Correction** usually includes: fixing character-level errors, whitespace issues, and garbled text.

Use the dataset version on Canvas.

Navigator models:
llama-3.1-70b-instruct
llama-3.1-8b-instruct
llama-3.1-nemotron-nano-8B-v1
llama-3.3-70b-instruct
mistral-7b-instruct
mistral-small-3.1
nemotron-3-nano-30b-a3b
codestral-22b
gemma-3-27b-it
gpt-oss-20b
gpt-oss-120b
granite-3.3-8b-instruct
sfr-embedding-mistral
nomic-embed-text-v1.5
flux.1-dev
flux.1-schnell
whisper-large-v3
kokoro

In [ ]:
%pip install openai lxml -q

In [ ]:
import requests
from google.colab import userdata

resp = requests.get(
    "https://api.anthropic.com/v1/models",
    headers={
        "x-api-key": userdata.get('TIC'),
        "anthropic-version": "2023-06-01"
    },
    params={"limit": 100}
)

for model in resp.json()["data"]:
    print(f"{model['id']:45s}  {model['display_name']}")

In [ ]:
import json, re, time
from pathlib import Path
import openai
from lxml import etree
from google.colab import userdata, files
from openai import OpenAI

# ========== CONFIGURE YOUR API ==========
# Uncomment ONE option:

#Option A: UF LiteLLM Proxy
#client = OpenAI(base_url="https://api.ai.it.ufl.edu", api_key=userdata.get('thinking_in_code'))
#MODEL = "gpt-oss-120b"

# Option B: Anthropic direct
client = OpenAI(base_url="https://api.anthropic.com/v1/", api_key=userdata.get('TIC'))
MODEL = "claude-opus-4-6"

print(f"Model: {MODEL}")

Model: claude-opus-4-6


In [ ]:
# ========== UPLOAD FILES ==========
uploaded = files.upload()
raw_texts = {k: v.decode('utf-8', errors='replace') for k, v in sorted(uploaded.items())}
print(f"Loaded {len(raw_texts)} files")
for name, text in raw_texts.items():
    print(f"  {name}: {len(text.split())} words")

Loaded 0 files


In [ ]:
# ========== A helper function to put system message and user message together for LLM calls==========
def call_llm(system, user, temperature=0.1):
    #Single LLM call with retry
    for attempt in range(3):
        try:
            r = client.chat.completions.create(
                model=MODEL, temperature=temperature, max_tokens=4096,
                messages=[{"role": "system", "content": system},
                          {"role": "user", "content": user}]
            )
            return r.choices[0].message.content
        except Exception as e:
            print(f"  Retry {attempt+1}: {e}")
            time.sleep(2 * (attempt + 1))
    raise RuntimeError("LLM call failed")

def parse_json(raw):
    #Parse JSON from LLM response, making a prettier output
    cleaned = re.sub(r'^```(?:json)?\s*', '', raw.strip())
    cleaned = re.sub(r'\s*```$', '', cleaned)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        print(f"  JSON parse failed, returning raw text")
        return {"corrected_text": raw, "changes": []}

##OCR Correction
One call per file. Returns corrected text + auditable changelog.

In [ ]:
OCR_SYSTEM = """You correct OCR errors in 1959 British colonial statutory instruments (Orders in Council) from Trinidad and Tobago's Royal Gazette.

Rules:
- Fix ONLY clear OCR errors: garbled chars, digit/letter confusion (l/1/I, 0/O, rn/m), broken words, bad line breaks.
- DO NOT modernize, rephrase, or restructure. Preserve original spelling and punctuation.
- Preserve all footnote markers and references.

Return ONLY a JSON object:
{"corrected_text": "...", "changes": [{"original": "...", "corrected": "...", "reason": "..."}]}
No markdown fences. No preamble."""

corrected = {}  # {filename: {"corrected_text": str, "changes": list}}

for fname, raw in raw_texts.items():
    print(f"Correcting {fname}...", end=" ")
    result = parse_json(call_llm(OCR_SYSTEM, raw))
    corrected[fname] = result
    print(f"{len(result['changes'])} fixes")
    time.sleep(0.5)

print(f"\nDone: {len(corrected)} files corrected.")

In [ ]:
# ========== REVIEW CHANGES ==========
for fname, result in corrected.items():
    print(f"\n--- {fname} ---")
    if not result['changes']:
        print("  No corrections.")
        continue
    for i, c in enumerate(result['changes'], 1):
        print(f"  {i}. \"{c.get('original','')}\" → \"{c.get('corrected','')}\"  ({c.get('reason','')})")